# MangoMAS Demo — Interactive Walkthrough

This notebook provides a hands-on tour of the **MangoMAS Multi-Agent Cognitive Architecture**.

The system brings together several components into a cohesive pipeline:

| Component | Description |
|---|---|
| **64-D Feature Extraction** | Deterministic vector encoding (32 hash + 16 domain + 8 structural + 4 sentiment + 4 novelty) |
| **10 Cognitive Cells** | Biologically-inspired processing units (Reasoning, Memory, Causal, Ethics, Empathy, Curiosity, FigLiteral, R2P, Telemetry, Aggregator) |
| **MCTS Planning** | Monte Carlo Tree Search with UCB1 and PUCT selection strategies, optional NN policy/value priors |
| **MoE Neural Router** | ~7M-parameter Mixture-of-Experts gate with semantic keyword boosting (~8 experts) |
| **8-Agent Orchestration** | Multi-agent task execution with three routing strategies: `moe_routing`, `round_robin`, `random` |

---

**Requirements:** Install the package with `pip install mangomas-demo`, or run this notebook from the repository root directory.

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────

import json

from mangomas_demo import (
    featurize64, plot_features,
    execute_cell, compose_cells,
    run_mcts, benchmark_strategies,
    route_task, orchestrate,
)
from mangomas_demo.cells.types import CELL_TYPES
from mangomas_demo.routing.router import EXPERT_NAMES
from mangomas_demo.agents.orchestrator import AGENTS

print("MangoMAS Demo loaded successfully.")
print(f"  Cell types available : {len(CELL_TYPES)}")
print(f"  Expert names         : {len(EXPERT_NAMES)}")
print(f"  Agents registered    : {len(AGENTS)}")

## 1. Feature Extraction

Every input string is converted into a **64-dimensional feature vector** before any downstream processing. The vector is L2-normalized and composed of five signal groups:

| Dims | Group | Description |
|------|-------|-------------|
| 0–31 | **Hash** (32) | SHA-256 sinusoidal content fingerprint |
| 32–47 | **Domain** (16) | Keyword presence flags (code, security, design, etc.) |
| 48–55 | **Structural** (8) | Length, punctuation density, word count, digit ratio |
| 56–59 | **Sentiment** (4) | Positive/negative keyword polarity and distance |
| 60–63 | **Novelty** (4) | Lexical diversity, line count, nesting, max word length |

This deterministic encoding ensures reproducible routing and cell behavior.

In [ ]:
# ── Extract features and inspect summary statistics ────────────────────────

SAMPLE_TEXT = (
    "Design a microservice architecture for a high-throughput e-commerce "
    "platform with security best practices and optimized caching layers."
)

features = featurize64(SAMPLE_TEXT)

print(f"Input text  : {SAMPLE_TEXT[:80]}...")
print(f"Vector dims : {len(features)}")
print(f"Min value   : {min(features):.6f}")
print(f"Max value   : {max(features):.6f}")
print(f"Mean value  : {sum(features) / len(features):.6f}")
print(f"L2 norm     : {sum(f * f for f in features) ** 0.5:.6f}")
print()
print("First 10 dimensions:")
for i, v in enumerate(features[:10]):
    print(f"  dim[{i:2d}] = {v:+.6f}")

In [ ]:
# ── Visualize feature vector with Plotly ───────────────────────────────────

fig = plot_features(features, title="64-D Feature Vector — Architecture + Security Input")
fig.show()

In [ ]:
# ── Compare features for two different inputs side-by-side ─────────────────

TEXT_A = "Write unit tests for the authentication module with full coverage."
TEXT_B = "Research recent advances in transformer architectures for NLP."

features_a = featurize64(TEXT_A)
features_b = featurize64(TEXT_B)

print(f"Text A: {TEXT_A}")
print(f"Text B: {TEXT_B}")
print()

# Cosine similarity
dot = sum(a * b for a, b in zip(features_a, features_b))
norm_a = sum(a * a for a in features_a) ** 0.5
norm_b = sum(b * b for b in features_b) ** 0.5
cosine_sim = dot / (norm_a * norm_b + 1e-8)

print(f"Cosine similarity: {cosine_sim:.4f}")
print(f"  (1.0 = identical, 0.0 = orthogonal)")
print()

# Show domain-tag differences (dims 32-47)
domain_tags = [
    "code", "function", "class", "api", "security", "threat",
    "architecture", "design", "data", "database", "test", "deploy",
    "optimize", "performance", "research", "analyze",
]
print("Domain tag activation differences:")
for i, tag in enumerate(domain_tags):
    va = features_a[32 + i]
    vb = features_b[32 + i]
    if va != vb:
        print(f"  {tag:>14s}:  A={va:.4f}  B={vb:.4f}")

## 2. Cognitive Cells

MangoMAS provides **10 biologically-inspired cognitive cells**, each with specialized processing heads:

| Cell | Name | Heads | Purpose |
|------|------|-------|---------|
| `reasoning` | ReasoningCell | rule, nn | Structured reasoning with configurable heads |
| `memory` | MemoryCell | preference_extractor | Privacy-preserving preference extraction |
| `causal` | CausalCell | do_calculus | Pearl's do-calculus for causal inference |
| `ethics` | EthicsCell | classifier, pii_scanner | Safety classification and PII detection |
| `empathy` | EmpathyCell | tone_detector | Emotional tone detection and empathetic response |
| `curiosity` | CuriosityCell | hypothesis_generator | Epistemic curiosity and question generation |
| `figliteral` | FigLiteralCell | classifier | Figurative vs literal language detection |
| `r2p` | R2PCell | planner | Requirements-to-Plan decomposition |
| `telemetry` | TelemetryCell | collector | Telemetry event capture and structuring |
| `aggregator` | AggregatorCell | weighted_average, max_confidence, ensemble | Multi-expert output aggregation |

In [ ]:
# ── List all registered cell types ─────────────────────────────────────────

print(f"{'Key':<14s} {'Name':<20s} {'Heads':<35s} Description")
print("-" * 100)
for key, info in CELL_TYPES.items():
    heads_str = ", ".join(info["heads"])
    print(f"{key:<14s} {info['name']:<20s} {heads_str:<35s} {info['description']}")

In [ ]:
# ── Execute the Reasoning cell ─────────────────────────────────────────────

REASONING_INPUT = (
    "We need to decompose the monolith into bounded contexts. "
    "The user service handles authentication and profile management. "
    "The order service manages cart operations and checkout flows."
)

reasoning_result = execute_cell("reasoning", REASONING_INPUT)

print("Reasoning Cell Output:")
print(json.dumps(reasoning_result, indent=2, default=str))

In [ ]:
# ── Execute the Ethics cell — PII detection & redaction ────────────────────

ETHICS_INPUT = (
    "Please process the account for John Doe. "
    "His email is john.doe@example.com and phone is 555-123-4567. "
    "SSN on file: 123-45-6789."
)

ethics_result = execute_cell("ethics", ETHICS_INPUT)

print("Ethics Cell Output:")
print(json.dumps(ethics_result, indent=2, default=str))
print()
print(f"Safe?          : {ethics_result.get('is_safe')}")
print(f"PII detected   : {len(ethics_result.get('pii_detected', []))} item(s)")
print(f"Redacted text  : {ethics_result.get('redacted_text')}")

In [ ]:
# ── Execute the Empathy cell — emotional tone detection ────────────────────

EMPATHY_INPUT = (
    "I'm really frustrated and stuck on this bug. "
    "I've been trying to fix it for hours and nothing works. "
    "It's overwhelming and I don't know what to do next."
)

empathy_result = execute_cell("empathy", EMPATHY_INPUT)

print("Empathy Cell Output:")
print(json.dumps(empathy_result, indent=2, default=str))
print()
print(f"Detected emotion   : {empathy_result.get('detected_emotion')}")
print(f"Confidence         : {empathy_result.get('confidence')}")
print(f"Empathetic response: {empathy_result.get('empathetic_response')}")

## 3. Cell Composition

Cells can be **chained into a pipeline** using `compose_cells()`. The function takes a comma-separated list of cell type names and executes them sequentially on the same input text.

Each cell's output is accumulated into a shared context, and the final result includes per-cell activation metadata (status, elapsed time) along with the last cell's full output.

In [ ]:
# ── Run a multi-cell pipeline ──────────────────────────────────────────────

PIPELINE_INPUT = (
    "Implement a secure login system with rate limiting. "
    "Consider potential SQL injection and XSS threat vectors. "
    "What are the underlying assumptions about user behavior?"
)
PIPELINE_SPEC = "reasoning, ethics, curiosity"

pipeline_result = compose_cells(PIPELINE_SPEC, PIPELINE_INPUT)

print(f"Pipeline: {pipeline_result.get('pipeline')}")
print(f"Total cells executed: {pipeline_result.get('total_cells')}")
print(f"Context keys accumulated: {pipeline_result.get('context_keys')}")
print()

print("Per-cell activations:")
for activation in pipeline_result.get("activations", []):
    print(f"  {activation['cell_type']:<12s}  status={activation['status']}  "
          f"elapsed={activation.get('elapsed_ms', 0):.2f} ms")

print()
print("Final output (last cell):")
print(json.dumps(pipeline_result.get("final_output", {}), indent=2, default=str))

## 4. MCTS Planning

The **Monte Carlo Tree Search (MCTS)** engine explores possible action sequences for a given task. It supports two selection strategies:

- **UCB1** (Upper Confidence Bound 1): Balances exploitation and exploration using `Q(s,a) + c * sqrt(ln(N) / n(s,a))`
- **PUCT** (Predictor + UCB for Trees): AlphaZero-style selection that incorporates neural network policy priors via `Q(s,a) + c * P(s,a) * sqrt(N) / (1 + n(s,a))`

When PyTorch is available, the engine uses a `PolicyNetwork` for action priors and a `ValueNetwork` for state evaluation. Otherwise it falls back to random rollouts.

Tasks are auto-categorized into: `architecture`, `implementation`, `optimization`, `security`, or `research`, each with domain-specific action sets.

In [ ]:
# ── Run MCTS with UCB1 strategy ────────────────────────────────────────────

MCTS_TASK = "Design a scalable microservice architecture for real-time analytics"

mcts_result = run_mcts(
    task=MCTS_TASK,
    max_simulations=100,
    exploration_constant=1.414,
    strategy="ucb1",
)

print(f"Task       : {mcts_result['task']}")
print(f"Category   : {mcts_result['category']}")
print(f"Strategy   : {mcts_result['strategy']}")
print(f"NN enabled : {mcts_result['nn_enabled']}")
print(f"Simulations: {mcts_result['total_simulations']}")
print(f"Best action: {mcts_result['best_action']}")
print(f"Best value : {mcts_result['best_value']}")
print(f"Elapsed    : {mcts_result['elapsed_ms']:.2f} ms")
print()

print("All actions ranked by visits:")
for action_info in mcts_result["all_actions"]:
    print(f"  {action_info['action']:<22s}  visits={action_info['visits']:>3d}  "
          f"value={action_info['value']:.3f}")

print()
print("Search tree (top 2 levels):")
print(json.dumps(mcts_result["tree"], indent=2, default=str))

In [ ]:
# ── Benchmark: MCTS vs Greedy vs Random ────────────────────────────────────

BENCHMARK_TASK = "Optimize database query performance for high-traffic endpoints"

bench = benchmark_strategies(BENCHMARK_TASK)

print(f"Task     : {bench['task']}")
print(f"Category : {bench['category']}")
print()

print(f"{'Strategy':<10s} {'Best Action':<25s} {'Quality':>8s} {'Time (ms)':>10s}")
print("-" * 58)
for strategy_name in ["mcts", "greedy", "random"]:
    r = bench["results"][strategy_name]
    print(f"{strategy_name:<10s} {r['best_action']:<25s} "
          f"{r['quality_score']:>8.3f} {r['elapsed_ms']:>10.2f}")

## 5. MoE Neural Routing

The **Mixture-of-Experts (MoE) Neural Router** selects the top-K most relevant experts for a given task. It combines:

1. A **~7M parameter neural gating network** (`RouterNet`) that maps the 64-dim feature vector to expert weights
2. **Keyword-based semantic boosting** that adds +0.15 weight per matching keyword to the relevant expert

The 8 experts are: *Code, Test, Design, Research, Architecture, Security, Performance, Docs*.

When PyTorch is not available, the router falls back to a deterministic feature-based routing scheme.

In [ ]:
# ── Route a security-related task ──────────────────────────────────────────

SECURITY_TASK = (
    "Perform a security audit on the authentication service. "
    "Check for OWASP Top 10 vulnerabilities and encrypt sensitive data at rest."
)

routing_result = route_task(SECURITY_TASK, top_k=3)

print(f"Task       : {routing_result['task'][:80]}...")
print(f"NN enabled : {routing_result['nn_enabled']}")
print(f"Top-K      : {routing_result['top_k']}")
print(f"Elapsed    : {routing_result['elapsed_ms']:.2f} ms")
print()

print("Selected experts:")
for expert in routing_result["selected_experts"]:
    print(f"  #{expert['rank']}  {expert['expert']:<22s}  weight={expert['weight']:.4f}")

print()
print("All expert weights:")
for name, weight in routing_result["all_weights"].items():
    bar = "#" * int(weight * 50)
    print(f"  {name:<22s}  {weight:.4f}  {bar}")

In [ ]:
# ── Route a code-related task and compare expert selection ─────────────────

CODE_TASK = (
    "Implement a Python function to parse CSV files "
    "and write unit tests with full coverage."
)

code_routing = route_task(CODE_TASK, top_k=3)

print(f"Task: {code_routing['task']}")
print()

print("Selected experts for code task:")
for expert in code_routing["selected_experts"]:
    print(f"  #{expert['rank']}  {expert['expert']:<22s}  weight={expert['weight']:.4f}")

print()
print("Comparison — Security task vs Code task (top-3 experts):")
print(f"  Security: {[e['expert'] for e in routing_result['selected_experts']]}")
print(f"  Code    : {[e['expert'] for e in code_routing['selected_experts']]}")

## 6. Agent Orchestration

The orchestration layer coordinates **8 specialized agents**, each mapped to a cognitive cell for real processing:

| Agent | Specialization | Cell |
|-------|---------------|------|
| SWE Agent | Code scaffold generation | reasoning |
| Architect Agent | System design and patterns | r2p |
| QA Agent | Test plan and case generation | reasoning |
| Security Agent | Threat modeling (OWASP) | ethics |
| DevOps Agent | Infrastructure planning | telemetry |
| Research Agent | Technical analysis | causal |
| Performance Agent | Optimization analysis | reasoning |
| Documentation Agent | Technical writing | reasoning |

Three routing strategies are available:
- **`moe_routing`** — Uses the MoE neural gate to select the most relevant agents
- **`round_robin`** — Selects agents sequentially from the list
- **`random`** — Randomly samples agents

In [ ]:
# ── Orchestrate with moe_routing strategy ─────────────────────────────────

ORCHESTRATION_TASK = (
    "Build a REST API with JWT authentication, "
    "input validation, and comprehensive test coverage."
)

orch_result = orchestrate(
    task=ORCHESTRATION_TASK,
    max_agents=3,
    strategy="moe_routing",
)

print(f"Task            : {orch_result['task']}")
print(f"Strategy        : {orch_result['strategy']}")
print(f"Agents selected : {orch_result['agents_selected']}")
print(f"Total elapsed   : {orch_result['total_elapsed_ms']:.2f} ms")
print()

for agent_result in orch_result["results"]:
    print(f"{agent_result['icon']} {agent_result['agent']}")
    print(f"    Specialization: {agent_result['specialization']}")
    print(f"    Cell used     : {agent_result['cell_used']}")
    print(f"    Weight        : {agent_result['weight']:.4f}")
    print(f"    Confidence    : {agent_result['confidence']}")
    print()

In [ ]:
# ── Compare all 3 orchestration strategies on the same task ────────────────

COMPARE_TASK = (
    "Analyze performance bottlenecks in the data pipeline "
    "and optimize throughput for real-time processing."
)

strategies = ["moe_routing", "round_robin", "random"]
strategy_results = {}

for strat in strategies:
    strategy_results[strat] = orchestrate(
        task=COMPARE_TASK,
        max_agents=3,
        strategy=strat,
    )

print(f"Task: {COMPARE_TASK}")
print()

for strat in strategies:
    res = strategy_results[strat]
    agents_used = [r["agent"] for r in res["results"]]
    avg_conf = (
        sum(r["confidence"] for r in res["results"]) / len(res["results"])
        if res["results"] else 0.0
    )
    print(f"Strategy: {strat}")
    print(f"  Agents   : {agents_used}")
    print(f"  Avg conf : {avg_conf:.3f}")
    print(f"  Elapsed  : {res['total_elapsed_ms']:.2f} ms")
    print()

## 7. End-to-End Pipeline

This section demonstrates the **full MangoMAS pipeline** in a single pass:

1. **Featurize** — Extract the 64-dim vector from the input task
2. **Route** — Use the MoE neural gate to select top-K experts
3. **Plan** — Run MCTS to find the best action sequence
4. **Execute** — Run cognitive cells based on expert-to-cell mapping
5. **Aggregate** — Combine results into a unified output

In [ ]:
# ── Full end-to-end pipeline ───────────────────────────────────────────────

E2E_TASK = (
    "Design and implement a secure API gateway with rate limiting, "
    "request validation, and comprehensive monitoring."
)

print("=" * 70)
print("STEP 1: Feature Extraction")
print("=" * 70)
e2e_features = featurize64(E2E_TASK)
print(f"  Vector dimensions: {len(e2e_features)}")
print(f"  L2 norm: {sum(f * f for f in e2e_features) ** 0.5:.6f}")
print(f"  Non-zero dims: {sum(1 for f in e2e_features if abs(f) > 0.001)}")
print()

print("=" * 70)
print("STEP 2: MoE Neural Routing")
print("=" * 70)
e2e_routing = route_task(E2E_TASK, top_k=3)
for expert in e2e_routing["selected_experts"]:
    print(f"  #{expert['rank']} {expert['expert']:<22s} weight={expert['weight']:.4f}")
print()

print("=" * 70)
print("STEP 3: MCTS Planning")
print("=" * 70)
e2e_mcts = run_mcts(E2E_TASK, max_simulations=50, strategy="ucb1")
print(f"  Category   : {e2e_mcts['category']}")
print(f"  Best action: {e2e_mcts['best_action']}")
print(f"  Best value : {e2e_mcts['best_value']}")
print()

print("=" * 70)
print("STEP 4: Cognitive Cell Execution")
print("=" * 70)
# Map selected experts to cell types and execute
expert_to_cell = {
    "Code Expert": "reasoning",
    "Test Expert": "reasoning",
    "Design Expert": "r2p",
    "Research Expert": "causal",
    "Architecture Expert": "r2p",
    "Security Expert": "ethics",
    "Performance Expert": "reasoning",
    "Docs Expert": "reasoning",
}
cell_outputs = []
for expert in e2e_routing["selected_experts"]:
    cell_type = expert_to_cell.get(expert["expert"], "reasoning")
    cell_result = execute_cell(cell_type, E2E_TASK)
    cell_outputs.append({
        "expert": expert["expert"],
        "cell": cell_type,
        "status": cell_result.get("status"),
        "elapsed_ms": cell_result.get("elapsed_ms"),
    })
    print(f"  {expert['expert']} -> {cell_type} cell: "
          f"status={cell_result['status']} ({cell_result.get('elapsed_ms', 0):.2f} ms)")
print()

print("=" * 70)
print("STEP 5: Aggregated Result")
print("=" * 70)
aggregate = {
    "task": E2E_TASK,
    "features_dim": len(e2e_features),
    "top_experts": [e["expert"] for e in e2e_routing["selected_experts"]],
    "mcts_best_action": e2e_mcts["best_action"],
    "mcts_best_value": e2e_mcts["best_value"],
    "cells_executed": cell_outputs,
    "pipeline_status": "completed",
}
print(json.dumps(aggregate, indent=2, default=str))

## Summary

This notebook demonstrated the core components of the **MangoMAS Multi-Agent Cognitive Architecture**:

1. **Feature Extraction** — Deterministic 64-dim vector encoding with 5 signal groups
2. **Cognitive Cells** — 10 specialized processing cells (reasoning, ethics, empathy, curiosity, etc.) with structured JSON outputs
3. **Cell Composition** — Multi-cell pipeline chaining for complex processing flows
4. **MCTS Planning** — Tree search with UCB1/PUCT strategies and optional neural network priors
5. **MoE Neural Routing** — Expert selection via neural gating network with semantic boosting
6. **Agent Orchestration** — 8-agent coordination with 3 routing strategies
7. **End-to-End Pipeline** — Full flow from featurization through routing, planning, execution, and aggregation

---

**Resources:**
- Documentation: See `README.md` in the repository root
- GitHub: [MangoMAS-Demo](https://github.com/mangomas/mangomas-demo)
- API Reference: Run `help(mangomas_demo)` for module-level documentation